# PMD Comment-to-Code Ratio — Raw Output Extraction (Java)

This notebook analyzes **Java repositories** with **PMD** and derives **Comment-to-Code Ratio** from extracted comment and code line metrics alongside PMD findings.

**Default benchmark repository:** [spring-projects/spring-petclinic](https://github.com/spring-projects/spring-petclinic)

> **Note:** Comment-to-Code Ratio is a **Derived** metric. PMD does not emit it directly; the notebook extracts comment/code lines from Java sources and combines them with PMD analysis output.


## Section 1 — Install Dependencies

Install Python packages, configure Java, and bootstrap PMD 7.14.0.


In [ ]:
!pip install -q pandas gitpython jupyter

import subprocess, sys
from pathlib import Path

METRIC_ROOT = Path('.').resolve()
PROJECT_ROOT = METRIC_ROOT
for _ in range(8):
    runtimes = PROJECT_ROOT / 'runtimes'
    if runtimes.is_dir() and (runtimes / 'jdk-21').is_dir():
        break
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        break
    PROJECT_ROOT = PROJECT_ROOT.parent

TOOL_ROOT = METRIC_ROOT / 'tool'
if str(TOOL_ROOT) not in sys.path:
    sys.path.insert(0, str(TOOL_ROOT))

from run_comment_to_code_ratio_benchmark_impl import configure_java_runtime, download_pmd

RUNTIMES_ROOT = PROJECT_ROOT / 'runtimes'
JDK_HOME = RUNTIMES_ROOT / 'jdk-21'
PMD_HOME = RUNTIMES_ROOT / 'pmd-bin-7.14.0'

configure_java_runtime(JDK_HOME)
download_pmd(PMD_HOME, cache_dir=RUNTIMES_ROOT / 'cache')

subprocess.run(['java', '-version'], check=False)
subprocess.run([str(PMD_HOME / 'bin' / ('pmd.bat' if sys.platform.startswith('win') else 'pmd')), 'check', '--help'], check=False)


## Section 2 — Configuration


In [ ]:
USE_GIT_URL = True

REPO_URL = 'https://github.com/spring-projects/spring-petclinic.git'

LOCAL_REPO_PATH = '/content/spring-petclinic'

WORKSPACE_DIR = './workspace'

OUTPUT_DIR = './outputs'

IF_CLONE_EXISTS = 'reuse'

CLONE_DEPTH = 1

RAW_OUTPUT_PREVIEW_LINES = 150

# Fast validation benchmark:
# USE_GIT_URL = False
# LOCAL_REPO_PATH = './workspace/comment_to_code_ratio_benchmark'


## Section 3 — Imports and Utility Functions


In [ ]:
from pathlib import Path
import sys

TOOL_ROOT = Path('tool').resolve()
if str(TOOL_ROOT) not in sys.path:
    sys.path.insert(0, str(TOOL_ROOT))

from __future__ import annotations

import csv
import shutil
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import pandas as pd
from IPython.display import display
from git import Repo
from git.exc import GitCommandError, InvalidGitRepositoryError

EXCLUDED_DIR_NAMES = {
    ".git", "target", "build", "out", "bin", ".gradle", ".mvn", "node_modules", "docs", "generated-sources",
}


class NotebookLogger:
    def __init__(self, error_log_path: Path) -> None:
        self.error_log_path = error_log_path
        self.error_log_path.parent.mkdir(parents=True, exist_ok=True)
        self._errors: list[dict[str, str]] = []
        if not self.error_log_path.exists():
            self.write_errors()

    def info(self, message: str) -> None:
        timestamp = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
        print(f"[{timestamp}] INFO: {message}")

    def error(self, message: str, file: str = "notebook") -> None:
        timestamp = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
        line = f"[{timestamp}] ERROR: {message}\n"
        print(line, end="")
        self._errors.append({"timestamp": timestamp, "file": file, "error_message": message})
        self.write_errors()

    def write_errors(self) -> None:
        with self.error_log_path.open("w", encoding="utf-8", newline="") as handle:
            writer = csv.DictWriter(handle, fieldnames=["timestamp", "file", "error_message"])
            writer.writeheader()
            writer.writerows(self._errors)


def ensure_output_dir(output_dir: Path) -> None:
    output_dir.mkdir(parents=True, exist_ok=True)


def derive_clone_path(repo_url: str, workspace_dir: Path) -> Path:
    repo_name = repo_url.rstrip("/").removesuffix(".git").split("/")[-1]
    if not repo_name:
        raise ValueError(f"Unable to derive repository name from URL: {repo_url}")
    return workspace_dir / repo_name


def validate_repo_url(repo_url: str) -> None:
    if not repo_url or not isinstance(repo_url, str):
        raise ValueError("REPO_URL must be a non-empty string.")
    if not (repo_url.startswith("https://") or repo_url.startswith("git@") or repo_url.startswith("http://")):
        raise ValueError(f"Invalid repository URL format: {repo_url}")


def clone_or_reuse_repository(
    repo_url: str, workspace_dir: Path, if_clone_exists: str, logger: NotebookLogger, clone_depth: int | None = None,
) -> Path:
    validate_repo_url(repo_url)
    workspace_dir.mkdir(parents=True, exist_ok=True)
    clone_path = derive_clone_path(repo_url, workspace_dir)
    if clone_path.exists():
        if if_clone_exists == "reclone":
            logger.info(f"Removing existing clone at {clone_path}")
            shutil.rmtree(clone_path)
        elif if_clone_exists == "reuse":
            logger.info(f"Reusing existing clone at {clone_path}")
            return clone_path.resolve()
        else:
            raise ValueError("IF_CLONE_EXISTS must be 'reuse' or 'reclone'.")
    logger.info(f"Cloning {repo_url} into {clone_path}")
    clone_kwargs: dict[str, Any] = {"depth": clone_depth} if clone_depth else {}
    try:
        Repo.clone_from(repo_url, clone_path, **clone_kwargs)
    except GitCommandError as exc:
        logger.error(f"Git clone failed: {exc}", file=repo_url)
        raise
    return clone_path.resolve()


def validate_local_repo_path(local_repo_path: Path, logger: NotebookLogger) -> Path:
    if not local_repo_path.exists():
        msg = f"Local repository path does not exist: {local_repo_path}"
        logger.error(msg, file=str(local_repo_path))
        raise FileNotFoundError(msg)
    if not local_repo_path.is_dir():
        msg = f"Local repository path is not a directory: {local_repo_path}"
        logger.error(msg, file=str(local_repo_path))
        raise NotADirectoryError(msg)
    try:
        Repo(local_repo_path)
        logger.info("Validated Git repository.")
    except InvalidGitRepositoryError:
        logger.info("Path is not a Git repository; proceeding as source directory.")
    return local_repo_path.resolve()


def resolve_repository_path(
    use_git_url: bool, repo_url: str, local_repo_path: str | Path, workspace_dir: Path,
    if_clone_exists: str, logger: NotebookLogger, clone_depth: int | None = None,
) -> Path:
    if use_git_url:
        return clone_or_reuse_repository(repo_url, workspace_dir, if_clone_exists, logger, clone_depth)
    return validate_local_repo_path(Path(local_repo_path), logger)


def discover_java_files(repo_path: Path) -> list[Path]:
    files: list[Path] = []
    for path in repo_path.rglob("*.java"):
        if any(part in EXCLUDED_DIR_NAMES for part in path.parts):
            continue
        files.append(path.resolve())
    return sorted(files)


def compute_repository_stats(repo_path: Path, java_files: list[Path]) -> dict[str, Any]:
    total_size = sum(path.stat().st_size for path in java_files)
    directories = {path.parent for path in java_files}
    return {
        "repository_name": repo_path.name,
        "repository_size_bytes": total_size,
        "directory_count": len(directories),
        "java_file_count": len(java_files),
    }


def save_java_inventory(java_files: list[Path], output_csv: Path) -> None:
    pd.DataFrame(
        [{"file_path": str(p), "file_name": p.name, "directory": str(p.parent)} for p in java_files]
    ).to_csv(output_csv, index=False)


def preview_raw_output(raw_text: str, preview_lines: int, output_path: Path) -> None:
    lines = raw_text.splitlines()
    print(f"\n{'=' * 80}")
    print(f"RAW PMD OUTPUT PREVIEW (first {preview_lines} lines)")
    print(f"{'=' * 80}\n")
    if not lines:
        print("(empty raw output)")
        return
    print("\n".join(lines[:preview_lines]))
    remaining = len(lines) - preview_lines
    if remaining > 0:
        print(f"\n... ({remaining} more lines saved to {output_path})")

from run_comment_to_code_ratio_benchmark_impl import (
    build_comment_code_metrics,
    build_maintainability_summary,
    combine_raw,
    compute_comment_ratio,
    merge_violations,
    parse_pmd_csv,
    parse_pmd_json,
    parse_pmd_text_violations,
    run_pmd,
    violations_from_csv,
    write_custom_ruleset,
)



## Section 4 — Repository Setup


In [ ]:
OUTPUT_PATH = Path(OUTPUT_DIR).resolve()
WORKSPACE_PATH = Path(WORKSPACE_DIR).resolve()
ERROR_LOG_PATH = OUTPUT_PATH / 'error_log.txt'

ensure_output_dir(OUTPUT_PATH)
logger = NotebookLogger(ERROR_LOG_PATH)

try:
    REPO_PATH = resolve_repository_path(
        use_git_url=USE_GIT_URL, repo_url=REPO_URL, local_repo_path=LOCAL_REPO_PATH,
        workspace_dir=WORKSPACE_PATH, if_clone_exists=IF_CLONE_EXISTS, logger=logger, clone_depth=CLONE_DEPTH,
    )
except Exception as exc:
    logger.error(f'Repository setup failed: {exc}')
    raise

JAVA_FILES = discover_java_files(REPO_PATH)
if not JAVA_FILES:
    logger.error('No Java source files found in repository.', file=str(REPO_PATH))
    raise FileNotFoundError('No Java source files found.')

REPO_STATS = compute_repository_stats(REPO_PATH, JAVA_FILES)
logger.info(f'Repository ready at: {REPO_PATH}')
print(f"Repository: {REPO_STATS['repository_name']}")
print(f"Size (Java files): {REPO_STATS['repository_size_bytes']:,} bytes")
print(f"Directories: {REPO_STATS['directory_count']:,}")
print(f"Java files: {REPO_STATS['java_file_count']:,}")


## Section 5 — Discover Java Files


In [ ]:
INVENTORY_CSV = OUTPUT_PATH / 'java_files_inventory.csv'
save_java_inventory(JAVA_FILES, INVENTORY_CSV)

print(f'Total Java Files Found: {len(JAVA_FILES)}')
print(f'Saved inventory to: {INVENTORY_CSV}')


## Section 6 — Generate PMD Ruleset


In [ ]:
RULESET_PATH = OUTPUT_PATH / 'custom_ruleset.xml'
write_custom_ruleset(RULESET_PATH)

print(f'Generated ruleset: {RULESET_PATH}')
print(RULESET_PATH.read_text(encoding='utf-8'))


## Section 7 — Execute PMD

Run PMD in text, JSON, and CSV formats. Preserve stdout/stderr exactly as emitted.


In [ ]:
PMD_CONSOLE_CHUNKS: list[str] = []

raw_out, raw_err, raw_code = run_pmd(PMD_HOME, REPO_PATH, RULESET_PATH, 'text')
csv_out, csv_err, csv_code = run_pmd(PMD_HOME, REPO_PATH, RULESET_PATH, 'csv')
json_out, json_err, json_code = run_pmd(PMD_HOME, REPO_PATH, RULESET_PATH, 'json')

PMD_CONSOLE_CHUNKS.append('===== pmd check (text) =====\n' + combine_raw(raw_out, raw_err))
PMD_CONSOLE_CHUNKS.append('===== pmd check (csv) =====\n' + combine_raw(csv_out, csv_err))
PMD_CONSOLE_CHUNKS.append('===== pmd check (json) =====\n' + combine_raw(json_out, json_err))

if raw_code not in {0, 4} and not raw_out.strip():
    logger.error(f'PMD text run exited with code {raw_code}', file='pmd_text')
if csv_code not in {0, 4} and not csv_out.strip():
    logger.error(f'PMD CSV run exited with code {csv_code}', file='pmd_csv')
if json_code not in {0, 4} and not json_out.strip():
    logger.error(f'PMD JSON run exited with code {json_code}', file='pmd_json')

logger.info('PMD execution complete.')


## Section 8 — Raw Output Extraction


In [ ]:
CONSOLE_PATH = OUTPUT_PATH / 'pmd_raw_console_output.txt'
JSON_PATH = OUTPUT_PATH / 'pmd_output.json'
CSV_PATH = OUTPUT_PATH / 'pmd_output.csv'

CONSOLE_PATH.write_text('\n'.join(PMD_CONSOLE_CHUNKS), encoding='utf-8')
JSON_PATH.write_text(json_out, encoding='utf-8')
CSV_PATH.write_text(csv_out, encoding='utf-8')

FINDINGS_DF = merge_violations(
    violations_from_csv(parse_pmd_csv(csv_out)),
    parse_pmd_json(json_out),
    parse_pmd_text_violations(raw_out),
)
FINDINGS_DF.to_csv(OUTPUT_PATH / 'pmd_findings.csv', index=False)

logger.info(f'Saved PMD outputs and {len(FINDINGS_DF)} findings.')
preview_raw_output(CONSOLE_PATH.read_text(encoding='utf-8'), RAW_OUTPUT_PREVIEW_LINES, CONSOLE_PATH)


## Section 9 — Extract Comment and Code Metrics

Parse Java source files for comment and executable code line counts.


In [ ]:
COMMENT_METRICS_DF = build_comment_code_metrics(JAVA_FILES)
COMMENT_METRICS_CSV = OUTPUT_PATH / 'comment_code_metrics.csv'
COMMENT_METRICS_DF.to_csv(COMMENT_METRICS_CSV, index=False)

logger.info(f'Extracted comment/code metrics for {len(COMMENT_METRICS_DF)} files.')
display(COMMENT_METRICS_DF)


## Section 10 — Comment-to-Code Ratio (Derived)

**Derived metric** (not emitted directly by PMD):

```text
Total_Comment_Lines = javadoc_lines + block_comment_lines + single_comment_lines
Comment_to_Code_Ratio = Total_Comment_Lines / code_lines
```


In [ ]:
COMMENT_RATIO = compute_comment_ratio(COMMENT_METRICS_DF)
RATIO_CSV = OUTPUT_PATH / 'comment_to_code_ratio_summary.csv'
pd.DataFrame([
    {'metric_name': 'Comment_to_Code_Ratio', 'metric_value': COMMENT_RATIO['comment_to_code_ratio']},
]).to_csv(RATIO_CSV, index=False)

logger.info(f"Comment-to-Code Ratio={COMMENT_RATIO['comment_to_code_ratio']} (Derived)")
display(pd.read_csv(RATIO_CSV))


## Section 11 — Comment Percentage (Derived)


In [ ]:
PERCENTAGE_CSV = OUTPUT_PATH / 'comment_percentage_summary.csv'
pd.DataFrame([
    {'metric_name': 'Comment_to_Code_Percentage', 'metric_value': COMMENT_RATIO['comment_to_code_percentage']},
]).to_csv(PERCENTAGE_CSV, index=False)

logger.info(f"Comment Percentage={COMMENT_RATIO['comment_to_code_percentage']}%")
display(pd.read_csv(PERCENTAGE_CSV))


## Section 12 — Maintainability Summary


In [ ]:
MAINTAINABILITY_DF = build_maintainability_summary(FINDINGS_DF)
MAINTAINABILITY_CSV = OUTPUT_PATH / 'maintainability_summary.csv'
MAINTAINABILITY_DF.to_csv(MAINTAINABILITY_CSV, index=False)

display(MAINTAINABILITY_DF)


## Section 13 — Summary Dashboard


In [ ]:
code_smells = int(MAINTAINABILITY_DF.loc[MAINTAINABILITY_DF['metric_name'] == 'Total_Code_Smells', 'metric_value'].iloc[0])
maint_violations = int(MAINTAINABILITY_DF.loc[MAINTAINABILITY_DF['metric_name'] == 'Maintainability_Violations', 'metric_value'].iloc[0])

summary_df = pd.DataFrame([
    {'Metric': 'Total Java Files', 'Value': len(JAVA_FILES)},
    {'Metric': 'Total Comment Lines', 'Value': int(COMMENT_RATIO['total_comment_lines'])},
    {'Metric': 'Total Code Lines', 'Value': int(COMMENT_RATIO['total_code_lines'])},
    {'Metric': 'Comment-to-Code Ratio (Derived)', 'Value': COMMENT_RATIO['comment_to_code_ratio']},
    {'Metric': 'Comment Percentage (Derived)', 'Value': COMMENT_RATIO['comment_to_code_percentage']},
    {'Metric': 'Code Smells', 'Value': code_smells},
    {'Metric': 'Maintainability Violations', 'Value': maint_violations},
])
display(summary_df)

deliverables = [
    CONSOLE_PATH, JSON_PATH, CSV_PATH, OUTPUT_PATH / 'pmd_findings.csv', COMMENT_METRICS_CSV,
    RATIO_CSV, PERCENTAGE_CSV, MAINTAINABILITY_CSV, INVENTORY_CSV, ERROR_LOG_PATH, RULESET_PATH,
]
print('\nDeliverables:')
for path in deliverables:
    print(f"  [{'OK' if path.exists() else 'MISSING'}] {path}")


## Section 14 — Error Handling


In [ ]:
if ERROR_LOG_PATH.exists() and ERROR_LOG_PATH.stat().st_size > 0:
    print(ERROR_LOG_PATH.read_text(encoding='utf-8'))
else:
    print('No errors logged.')


## Section 15 — Deliverables

```text
outputs/
├── pmd_raw_console_output.txt
├── pmd_output.json
├── pmd_output.csv
├── pmd_findings.csv
├── comment_code_metrics.csv
├── comment_to_code_ratio_summary.csv
├── comment_percentage_summary.csv
├── maintainability_summary.csv
├── java_files_inventory.csv
├── custom_ruleset.xml
└── error_log.txt
```
